In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os

dataset_path = "/kaggle/input"

print("Available datasets:")
print(os.listdir(dataset_path))


In [ ]:

import os

print(os.listdir("/kaggle/input/competitions"))


In [ ]:
from pathlib import Path

base = Path("/kaggle/input/competitions")

corridors = list(base.glob("*/kaggle_public/corridors"))
corridors += list(base.glob("*/corridors"))

if corridors:
    corridor_path = corridors[0]
    print("Dataset location:", corridor_path)
    print("\nAvailable freeway corridors:")
    for folder in sorted(corridor_path.iterdir()):
        if folder.is_dir():
            print(" -", folder.name)
else:
    print("Corridors folder not found.")


Avance — Dataset Exploration & Initial Traffic Analysis (EDA)

In [ ]:
import pandas as pd

# Select one day of traffic data from I-5 North
file = (
    folder
    / "train/mainline_states/year_month=2030-06/"
    / "synthetic_mainline_2030_06_01.parquet"
)

# Load the dataset
df = pd.read_parquet(file)

# Display the first 5 observations
print("Dataset shape:", df.shape)
display(df.head())


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt

# Prepare data
df["timestamp"] = pd.to_datetime(df["timestamp"])

# Select the sensor with the most speed records
sensor_id = (
    df.groupby("station_id")["speed_kmh"]
    .count()
    .idxmax()
)

sensor_data = df[
    df["station_id"] == sensor_id
].sort_values("timestamp")

# Plot traffic speed throughout the day
plt.figure(figsize=(12, 5))

plt.plot(
    sensor_data["timestamp"],
    sensor_data["speed_kmh"],
    color="royalblue"
)

plt.title(f"Traffic Speed Over Time — {sensor_id}")
plt.xlabel("Time")
plt.ylabel("Speed (km/h)")
plt.grid(alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("Selected sensor:", sensor_id)
print("Total records:", len(sensor_data))


### Initial Finding — Traffic Speed Analysis

We successfully loaded and visualized synthetic traffic measurements from the TrafficFlowBench dataset for the D12_I5_N corridor.

Our initial analysis focused on sensor SYN_D12_I5_N_0131, with 288 records collected at five-minute intervals on June 1, 2030.

The recorded speed remained relatively stable, approximately between 114 and 120 km/h, with no sustained speed reduction evident during the selected day.

This initial exploration demonstrates successful dataset access and provides a starting point for investigating missing observations and traffic state reconstruction.

Can we reconstruct missing traffic measurements?

In [ ]:

# Inspect observation and missing-data indicators

for column in ["is_observed", "is_imputed", "is_missing"]:
    print(f"\n{column}:")
    print(sensor_data[column].value_counts(dropna=False))


In [ ]:

# Examine how observation and imputation flags overlap
print("Observation vs. Imputation:")
display(pd.crosstab(
    sensor_data["is_observed"],
    sensor_data["is_imputed"],
    margins=True
))

# Inspect five records marked as missing
print("\nExamples of missing records:")
display(
    sensor_data.loc[
        sensor_data["is_missing"] == 1,
        ["timestamp", "speed_kmh", "flow_vph",
         "is_observed", "is_imputed", "is_missing"]
    ].head()
)


In [ ]:

import numpy as np
import pandas as pd

# Sort measurements chronologically
data = sensor_data.sort_values("timestamp").reset_index(drop=True).copy()

# Select observed, non-imputed speeds
valid = np.flatnonzero(
    (data["is_observed"] == 1) &
    (data["is_imputed"] == 0) &
    data["speed_kmh"].notna()
)

# Randomly hide 20% of eligible measurements
rng = np.random.default_rng(42)
hidden = rng.choice(
    valid,
    size=max(1, int(len(valid) * 0.20)),
    replace=False
)

actual = data["speed_kmh"].iloc[hidden].copy()

# Remove selected values temporarily
masked_speed = data["speed_kmh"].copy()
masked_speed.iloc[hidden] = np.nan

# Reconstruct with linear interpolation
predicted = masked_speed.interpolate(
    method="linear",
    limit_direction="both"
)

# Evaluate only artificially hidden values
mae = np.mean(
    np.abs(actual.to_numpy() - predicted.iloc[hidden].to_numpy())
)

print("First Reconstruction Baseline")
print("Measurements hidden:", len(hidden))
print(f"Mean Absolute Error: {mae:.4f} km/h")

results = pd.DataFrame({
    "Actual Speed": actual.to_numpy(),
    "Predicted Speed": predicted.iloc[hidden].to_numpy()
})

display(results.head(10))
